### Load Library

In [6]:
import pandas as pd
import glob
import os

### Load Data

In [8]:
annotation_dir = "data/annotations/"
metadata_file = "data/clean_metadata.tsv"
assaydata_file = "data/clean_assaydata.tsv"
output_file = "data/master_functional_matrix.tsv"

print("Loading clean metadata...")

metadata_df = pd.read_csv(metadata_file, sep='\t')
assaydata_df = pd.read_csv(assaydata_file, sep='\t')
print(f"Metadata loaded successfully. Found {len(metadata_df)} samples.")

display(metadata_df.head())
print("Loading annotation files...")
display(assaydata_df.head())

Loading clean metadata...
Metadata loaded successfully. Found 32 samples.


,Sample_ID,Sample_Type,Location,Collection_Date,Prep_Number
0,D1S1,Environmental,Floor by gowning room,27-May-2025,"1,2,3"
1,D1S2,Environmental,Floor by airlock,27-May-2025,"1,2,3"
2,D1S3,Environmental,Floor by main door,27-May-2025,"1,2,3"
3,D1S4,Environmental,Floor by cabinets,27-May-2025,"1,2,3"
4,D1S5,Environmental,Field Control,27-May-2025,"1,2,3"


Loading annotation files...


,Sample_ID,DNA_Concentration_ng_ul,Total_Reads,Human_Reads_Removed_Pct,Blank_Contaminant_Removed_Pct
0,D1S1,36.60,2312732,1.77,0.02
1,D1S2,93.40,2199286,1.09,0.01
2,D1S3,40.60,1671326,5.96,0.05
3,D1S4,45.80,1156137,6.71,0.09
4,D1S5,1.41,149239,31.58,0.58


In [9]:
# Find all downloaded TSV annotation files
file_pattern = os.path.join(annotation_dir, "*gene-coverage-annotation-and-tax.tsv")
annotation_files = glob.glob(file_pattern)

print(f"Found {len(annotation_files)} annotation files. Processing and concatenating...")

Found 25 annotation files. Processing and concatenating...


### Build Matrix

In [10]:
df_list = []

# Updated columns directly matching the NASA outputs
target_cols = ['gene_ID', 'coverage', 'KO_ID', 'KO_function', 'genus', 'species']

for file in annotation_files:

    # Extract the Sample_ID from the filename
    filename = os.path.basename(file)
    sample_id = filename.split('Prep3_')[1].split('-gene')[0]

    temp_df = pd.read_csv(file, sep='\t', usecols=target_cols)
    temp_df['Sample_ID'] = sample_id # Add Sample_ID column to the dataframe
    df_list.append(temp_df)

display(df_list[0].head())

,gene_ID,coverage,KO_ID,KO_function,genus,species,Sample_ID
0,c_D1S1_1_1,2.0000,NaN,NaN,Brevundimonas,NaN,D1S1
1,c_D1S1_3_1,6.2600,NaN,NaN,NaN,NaN,D1S1
2,c_D1S1_4_1,7.9190,NaN,NaN,NaN,NaN,D1S1
3,c_D1S1_6_1,8.6075,NaN,NaN,Paracoccus,NaN,D1S1
4,c_D1S1_10_1,7.8792,NaN,NaN,Vibrio,Vibrio cholerae,D1S1


In [11]:
# Concatenate all dataframes into one massive dataframe
print("Concatenating all genes into a single matrix...")
master_df = pd.concat(df_list, ignore_index=True)

# Clean the KO_ID column
print("Filtering out genes without KO_ID annotations...")
master_df = master_df.dropna(subset=['KO_ID'])

master_df.head()

Concatenating all genes into a single matrix...
Filtering out genes without KO_ID annotations...


,gene_ID,coverage,KO_ID,KO_function,genus,species,Sample_ID
6,c_D1S1_13_1,5.8662,K03183,demethylmenaquinone methyltransferase / 2-meth...,Brevundimonas,NaN,D1S1
14,c_D1S1_20_1,12.5448,K07112,uncharacterized protein,Pseudomonas,NaN,D1S1
15,c_D1S1_20_2,7.3931,K22042,"ArsR family transcriptional regulator, virulen...",Pseudomonas,NaN,D1S1
21,c_D1S1_26_2,1.8889,K16270,"3-methylcatechol 2,3-dioxygenase [EC:1.13.11.-]",NaN,NaN,D1S1
22,c_D1S1_27_1,16.7747,K00826,branched-chain amino acid aminotransferase [EC...,Brevundimonas,Brevundimonas bullata,D1S1


### Merge and Save

In [12]:
import pandas as pd

print("Merging functional data with physical cleanroom locations...")

final_df = master_df.merge(metadata_df[['Sample_ID', 'Location', 'Sample_Type']], on='Sample_ID', how='inner') \
                    .merge(assaydata_df[['Sample_ID', 'Total_Reads']], on='Sample_ID', how='inner')

# Normalize the data (Calculate CPM)
final_df['CPM'] = (final_df['coverage'] / final_df['Total_Reads']) * 1000000

print(final_df.shape)
final_df.head()

Merging functional data with physical cleanroom locations...
(62195, 11)


,gene_ID,coverage,KO_ID,KO_function,genus,species,Sample_ID,Location,Sample_Type,Total_Reads,CPM
0,c_D1S1_13_1,5.8662,K03183,demethylmenaquinone methyltransferase / 2-meth...,Brevundimonas,NaN,D1S1,Floor by gowning room,Environmental,2312732,2.536481
1,c_D1S1_20_1,12.5448,K07112,uncharacterized protein,Pseudomonas,NaN,D1S1,Floor by gowning room,Environmental,2312732,5.424234
2,c_D1S1_20_2,7.3931,K22042,"ArsR family transcriptional regulator, virulen...",Pseudomonas,NaN,D1S1,Floor by gowning room,Environmental,2312732,3.196696
3,c_D1S1_26_2,1.8889,K16270,"3-methylcatechol 2,3-dioxygenase [EC:1.13.11.-]",NaN,NaN,D1S1,Floor by gowning room,Environmental,2312732,0.816740
4,c_D1S1_27_1,16.7747,K00826,branched-chain amino acid aminotransferase [EC...,Brevundimonas,Brevundimonas bullata,D1S1,Floor by gowning room,Environmental,2312732,7.253197


In [13]:
# Save the master matrix
final_df.to_csv(output_file, sep='\t', index=False)